In [4]:
import sys
print(sys.executable)

c:\sk-encore\1st project\.venv\Scripts\python.exe


In [1]:
from selenium import webdriver

In [10]:
"""
서울특별시 장애인 복지 / 서울특별시 장애인 지원 / 서울특별시 장애인 콜택시
- 네이버 통합검색 → 뉴스 탭 → 무한 스크롤
- '네이버뉴스' 라벨이 붙은 기사만 수집 (언론사 자체 사이트 링크 제외)
- 헤드라인에 '장애인'이 포함된 기사만 저장
- 수집 항목: 헤드라인 / 본문 요약 / 언론사명 / URL
"""

import time
import random
import csv
from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager


KEYWORDS = [
    ("서울특별시 장애인 복지", "복지지원사업"),
    ("서울특별시 장애인 지원", "복지지원사업"),
    ("서울특별시 장애인 콜택시", "장애인콜택시"),
]

TARGET_COUNT = 100
MAX_SCROLL_NO_CHANGE = 6
TIME_LIMIT_PER_KEYWORD = 5 * 60  # 키워드당 최대 5분, 넘으면 진행 상황과 무관하게 다음 키워드로 이동


# ── 카드 하나 = 헤드라인/본문을 담은 블록(m9EMVjFAOiEZp87Q)의 '직속 부모' ──
# fdr-xxxxxxxx, div:nth-child(N) 은 검색할 때마다 랜덤/가변이라 사용하지 않고,
# 클래스명이 고정된 요소를 기준으로 상대 탐색한다.
TEXT_BLOCK_SEL = "div.m9EMVjFAOiEZp87Q"
HEADLINE_SEL = "span.sds-comps-text-type-headline1"
BODY_SEL = "span.sds-comps-text-type-body1"
PROFILE_BLOCK_SEL = "div.sds-comps-profile"
SOURCE_LINK_SEL = "div.sds-comps-profile-info-subtexts span:nth-child(4) > a"
SOURCE_LABEL_SEL = "span.sds-comps-text-weight-sm"
PRESS_NAME_SEL = "div.sds-comps-profile-info-title span > a > span.sds-comps-text-weight-sm"

CSV_FIELDS = ["category", "keyword", "headline", "body", "press", "url", "crawled_at"]


def build_driver():
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)


def go_to_news_tab(driver, keyword):
    """
    news.naver.com 검색 아이콘 → 입력창 선택자를 확신할 수 없어 계속 실패했으므로,
    검색 결과 URL로 직접 진입하는 방식으로 변경.
    (news.naver.com 검색창에 입력해도 최종적으로 도착하는 곳이 바로 이 URL과 동일함)
    대신 결과 대기 조건은 실제로 확인된 최신 카드 클래스(TEXT_BLOCK_SEL)로 지정.
    """
    url = f"https://search.naver.com/search.naver?where=news&query={keyword}"
    driver.get(url)

    try:
        WebDriverWait(driver, 8).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, TEXT_BLOCK_SEL))
        )
        time.sleep(random.uniform(1, 1.5))
        return True
    except TimeoutException:
        print(f"[{keyword}] 검색 결과 카드가 로드되지 않음 (TEXT_BLOCK_SEL 재확인 필요)")
        return False


def parse_card(text_block):
    """
    텍스트 블록(헤드라인+본문 컨테이너) 하나를 기준으로,
    같은 카드 안의 프로필 블록(언론사/URL)까지 함께 추출.
    → '직속 부모'를 카드 컨테이너로 삼아서, 다른 카드의 정보가 섞이는 문제를 방지.
    """
    try:
        headline = text_block.find_element(By.CSS_SELECTOR, HEADLINE_SEL).text.strip()
    except NoSuchElementException:
        return None

    if not headline or "장애인" not in headline:
        return None

    try:
        body = text_block.find_element(By.CSS_SELECTOR, BODY_SEL).text.strip()
    except NoSuchElementException:
        body = ""

    # 카드 컨테이너 = 텍스트 블록의 직속 부모 (헤드라인/본문/프로필이 모두 이 안에 함께 있음)
    try:
        card = text_block.find_element(By.XPATH, "..")
        profile_block = card.find_element(By.CSS_SELECTOR, PROFILE_BLOCK_SEL)
    except NoSuchElementException:
        print(f"[DEBUG] 프로필 블록 못 찾음 ({headline[:20]}...)")
        return None

    try:
        source_anchor = profile_block.find_element(By.CSS_SELECTOR, SOURCE_LINK_SEL)
        source_label = source_anchor.find_element(By.CSS_SELECTOR, SOURCE_LABEL_SEL).text.strip()
    except NoSuchElementException:
        return None

    # '네이버뉴스' 라벨이 붙은 기사만 수집 (언론사 자체 사이트 링크는 제외)
    if source_label != "네이버뉴스":
        return None

    url = source_anchor.get_attribute("href")
    if not url:
        return None

    try:
        press = profile_block.find_element(By.CSS_SELECTOR, PRESS_NAME_SEL).text.strip()
    except NoSuchElementException:
        press = ""

    return headline, body, press, url


def wait_for_more_cards(driver, prev_raw_count, timeout=5.0, poll_interval=0.5):
    """
    스크롤 직후 실제로 새 카드가 DOM에 추가될 때까지 짧게 폴링하며 기다린다.
    고정 sleep만 쓰면 로딩이 느릴 때 '아직 안 늘어난 상태'를 그냥 지나쳐버려서
    실제로는 더 로드될 카드가 있는데도 조기에 no_change로 판정되는 문제가 생긴다.
    """
    waited = 0.0
    while waited < timeout:
        time.sleep(poll_interval)
        waited += poll_interval
        try:
            new_raw_count = len(driver.find_elements(By.CSS_SELECTOR, TEXT_BLOCK_SEL))
        except WebDriverException:
            break
        if new_raw_count > prev_raw_count:
            return new_raw_count
    try:
        return len(driver.find_elements(By.CSS_SELECTOR, TEXT_BLOCK_SEL))
    except WebDriverException:
        return prev_raw_count


def crawl_keyword(driver, keyword, category, seen_urls, result):
    if not go_to_news_tab(driver, keyword):
        return

    collected = 0
    no_change_count = 0
    last_raw_count = 0  # 화면에 로드된 '전체 카드 개수' 기준 (필터링된 collected와는 다름)
    start_time = time.time()

    while collected < TARGET_COUNT and no_change_count < MAX_SCROLL_NO_CHANGE:
        # 5분 넘으면 진행 상황과 무관하게 이 키워드는 중단하고 다음으로 넘어감
        if time.time() - start_time > TIME_LIMIT_PER_KEYWORD:
            print(f"[{category}|{keyword}] 시간 제한(5분) 도달 → 다음 키워드로 이동")
            break

        try:
            text_blocks = driver.find_elements(By.CSS_SELECTOR, TEXT_BLOCK_SEL)
        except WebDriverException as e:
            print(f"[{category}|{keyword}] 카드 목록 조회 중 오류 발생, 재시도: {e}")
            time.sleep(2)
            continue

        raw_count = len(text_blocks)

        if not text_blocks:
            print(f"[{category}|{keyword}] 뉴스 카드 자체를 못 찾음 (셀렉터 확인 필요)")

        for block in text_blocks:
            try:
                parsed = parse_card(block)
            except (StaleElementReferenceException, WebDriverException) as e:
                # 스크롤 도중 DOM이 바뀌어 요소가 사라진 경우 → 이 카드만 건너뛰고 계속 진행
                print(f"[{category}|{keyword}] 카드 파싱 중 오류, 건너뜀: {type(e).__name__}")
                continue

            if parsed is None:
                continue
            headline, body, press, url = parsed

            if url in seen_urls:
                continue

            seen_urls.add(url)
            result.append({
                "category": category,
                "keyword": keyword,
                "headline": headline,
                "body": body,
                "press": press,
                "url": url,
                "crawled_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            })
            collected += 1

            if collected >= TARGET_COUNT:
                break

        print(f"[{category}|{keyword}] 누적 수집 {collected}건 / 현재 로드된 카드 {raw_count}개 (경과 {int(time.time() - start_time)}초)")

        # ── 핵심 수정 지점 ──
        # 예전 코드는 '조건에 맞는 기사 수(collected)'가 그대로면 no_change_count를 올렸다.
        # 하지만 스크롤로 새 카드가 계속 로드되고 있어도, 그 카드들 헤드라인에
        # '장애인'이 없으면 collected는 안 늘어난다 → 아직 더 볼 게 있는데 조기 중단됨.
        # 그래서 '화면에 실제로 로드된 전체 카드 개수(raw_count)'가 늘었는지로 판단해야
        # "페이지가 진짜로 더 이상 안 불러와진다"를 정확히 감지할 수 있다.
        if raw_count == last_raw_count:
            no_change_count += 1
        else:
            no_change_count = 0
        last_raw_count = raw_count

        if collected >= TARGET_COUNT:
            break

        try:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        except WebDriverException:
            pass

        # 고정 sleep 대신, 새 카드가 실제로 늘어날 때까지 최대 5초 폴링
        last_raw_count = wait_for_more_cards(driver, raw_count, timeout=5.0, poll_interval=0.5)
        time.sleep(random.uniform(0.5, 1.0))

    reason = "목표 건수 도달" if collected >= TARGET_COUNT else "더 이상 새 기사 없음 / 시간 제한"
    print(f"[{category}|{keyword}] 수집 종료 ({reason}) - 총 {collected}건\n")


def save_csv(filename, rows, write_header):
    mode = "w" if write_header else "a"
    with open(filename, mode, newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        if write_header:
            writer.writeheader()
        writer.writerows(rows)


def main():
    driver = build_driver()
    seen_urls = set()
    filename = f"disability_news_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    total_saved = 0

    try:
        for keyword, category in KEYWORDS:
            print(f"\n=== '{keyword}' ({category}) 수집 시작 ===")
            result = []  # 이번 키워드에서 새로 모은 것만 담는 리스트
            try:
                crawl_keyword(driver, keyword, category, seen_urls, result)
            finally:
                # 이 키워드에서 무슨 일이 있었든(정상 종료/시간초과/예외) 지금까지 모은 건 바로 저장
                if result:
                    save_csv(filename, result, write_header=(total_saved == 0))
                    total_saved += len(result)
                    print(f"[{category}|{keyword}] {len(result)}건 CSV에 저장 완료 (누적 {total_saved}건)")
    finally:
        driver.quit()

    print(f"\n총 {total_saved}건 저장 완료 → {filename}")


if __name__ == "__main__":
    main()


=== '서울특별시 장애인 복지' (복지지원사업) 수집 시작 ===
[복지지원사업|서울특별시 장애인 복지] 누적 수집 0건 / 현재 로드된 카드 10개 (경과 0초)
[복지지원사업|서울특별시 장애인 복지] 누적 수집 0건 / 현재 로드된 카드 20개 (경과 2초)
[복지지원사업|서울특별시 장애인 복지] 누적 수집 1건 / 현재 로드된 카드 30개 (경과 5초)
[복지지원사업|서울특별시 장애인 복지] 누적 수집 5건 / 현재 로드된 카드 40개 (경과 8초)
[복지지원사업|서울특별시 장애인 복지] 누적 수집 7건 / 현재 로드된 카드 50개 (경과 13초)
[복지지원사업|서울특별시 장애인 복지] 누적 수집 8건 / 현재 로드된 카드 60개 (경과 19초)
[복지지원사업|서울특별시 장애인 복지] 누적 수집 9건 / 현재 로드된 카드 70개 (경과 25초)
[복지지원사업|서울특별시 장애인 복지] 수집 종료 (더 이상 새 기사 없음 / 시간 제한) - 총 9건

[복지지원사업|서울특별시 장애인 복지] 9건 CSV에 저장 완료 (누적 9건)

=== '서울특별시 장애인 지원' (복지지원사업) 수집 시작 ===
[복지지원사업|서울특별시 장애인 지원] 누적 수집 4건 / 현재 로드된 카드 10개 (경과 0초)
[복지지원사업|서울특별시 장애인 지원] 누적 수집 4건 / 현재 로드된 카드 20개 (경과 2초)
[복지지원사업|서울특별시 장애인 지원] 누적 수집 6건 / 현재 로드된 카드 30개 (경과 6초)
[복지지원사업|서울특별시 장애인 지원] 누적 수집 9건 / 현재 로드된 카드 40개 (경과 11초)
[복지지원사업|서울특별시 장애인 지원] 누적 수집 12건 / 현재 로드된 카드 50개 (경과 16초)
[복지지원사업|서울특별시 장애인 지원] 누적 수집 12건 / 현재 로드된 카드 60개 (경과 21초)
[복지지원사업|서울특별시 장애인 지원] 누적 수집 13건 / 현재 로드된 카드 70개 (경과 26초)
[복지지원사업|서울특별시 장애인 지원] 수집 종료 (더 이상 새 기사 

In [ ]:

driver.quit()

AttributeError: 'ZMQExitAutocall' object has no attribute 'driver'